# Stacking Ensemble
Weighted ensemble combining all tree-based and deep learning models.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.ndimage import uniform_filter1d
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
import tensorflow as tf
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import warnings, glob
warnings.filterwarnings('ignore')

# Global constants
EXTREME_T   = 28        # Extreme heat threshold (°C)
MAN_LAT     = 53.48
MAN_LON_360 = 360 - 2.24  # Manchester longitude in 0-360 format


In [ ]:
# ─── Global plot settings (journal style) ───────────────────────────────────
matplotlib.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.titlesize':   12,
    'axes.labelsize':   11,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
    'legend.fontsize':  9,
    'figure.dpi':       130,
    'savefig.dpi':      300,
    'savefig.bbox':     'tight',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.22,
    'grid.linestyle':   '--',
    'axes.axisbelow':   True,
})

C = dict(
    RF='#2166AC', XGB='#4DAC26', LGB='#E08214',
    LSTM='#762A83', CNN='#D6604D', CNN_LSTM='#01665E',
    Ensemble='#543005', obs='#252525',
    hist='#4393C3', fut='#D6604D', extreme='#B2182B',
)

def savefig(name):
    plt.savefig(f'{name}.png', dpi=300, bbox_inches='tight', facecolor='white')


## Prerequisites
Run `1_data_processing.ipynb`, `3_LSTM_prediction.ipynb`, and `4_RF_prediction.ipynb` first so that all model predictions are available.

In [153]:
# =============================================================================
# PART 7  集成模型（按 R² 加权）
# =============================================================================
print('\n' + '=' * 70)
print('PART 7 — Weighted Ensemble')
print('=' * 70)
 
# 对齐长度：DL 序列比树模型短 LOOKBACK 个
n_seq = len(y_te_seq_C)
rf_al  = y_pred_rf_C[-n_seq:]
xgb_al = y_pred_xgb_C[-n_seq:]
lgb_al = y_pred_lgb_C[-n_seq:]
y_true_ens = y_te_seq_C   # 共同真值（原始°C）
 
# 计算各模型 R²（在对齐的测试集上）
r2_scores = {
    'RF':       r2_score(y_true_ens, rf_al),
    'XGB':      r2_score(y_true_ens, xgb_al),
    'LGB':      r2_score(y_true_ens, lgb_al),
    'LSTM':     r2_score(y_true_ens, y_pred_lstm_C),
    'CNN':      r2_score(y_true_ens, y_pred_cnn_C),
    'CNN-LSTM': r2_score(y_true_ens, y_pred_cnnlstm_C),
}
 
# 负R²的模型权重置0（避免拉低集成效果）
r2_pos   = {k: max(v, 0) for k, v in r2_scores.items()}
total_r2 = sum(r2_pos.values()) + 1e-9
weights  = {k: v / total_r2 for k, v in r2_pos.items()}
print('  Ensemble weights (R²-based):')
for k, v in weights.items(): print(f'    {k:<12} {v:.4f}')
 
y_pred_ens_C = (
    weights['RF']       * rf_al +
    weights['XGB']      * xgb_al +
    weights['LGB']      * lgb_al +
    weights['LSTM']     * y_pred_lstm_C +
    weights['CNN']      * y_pred_cnn_C +
    weights['CNN-LSTM'] * y_pred_cnnlstm_C
)
ens_metrics = evaluate_C(y_true_ens, y_pred_ens_C, 'Ensemble')
ens_ext     = extreme_scores(y_true_ens, y_pred_ens_C)
 
all_metrics_df = pd.concat([tree_metrics_df, dl_metrics_df,
                             pd.DataFrame({'Ensemble': ens_metrics}, index=['MAE','RMSE','R2','MAPE']).T])
print('\nAll models (°C):')
print(all_metrics_df.round(4).to_string())
 
# 极端事件汇总
ext_summary = pd.DataFrame({
    'RF': rf_ext, 'XGB': xgb_ext, 'LGB': lgb_ext,
    'LSTM': lstm_ext, 'CNN': cnn_ext, 'CNN-LSTM': cnnlstm_ext,
    'Ensemble': ens_ext,
}).T[['POD','FAR','TS']]
print('\nExtreme event scores:')
print(ext_summary.round(3).to_string())
 
print('\n✅ All models trained and evaluated. Now generating research figures...\n')
 


PART 7 — Weighted Ensemble
  Ensemble weights (R²-based):
    RF           0.1774
    XGB          0.1815
    LGB          0.1817
    LSTM         0.1483
    CNN          0.1543
    CNN-LSTM     0.1568
  Ensemble           MAE=1.010°C  RMSE=1.328°C  R²=0.9423  MAPE=6.45%

All models (°C):
                  MAE    RMSE      R2     MAPE
Random Forest  0.8142  1.0953  0.9607   5.0959
XGBoost        0.5306  0.7288  0.9826   3.4162
LightGBM       0.5078  0.7001  0.9839   3.2679
LSTM           1.8790  2.4519  0.8032  11.8240
CNN            1.7707  2.2429  0.8353  11.5201
CNN-LSTM       1.6722  2.1482  0.8490  10.6632
Ensemble       1.0103  1.3279  0.9423   6.4524

Extreme event scores:
            POD    FAR     TS
RF        0.287  0.065  0.282
XGB       0.802  0.129  0.717
LGB       0.802  0.120  0.723
LSTM      0.000  0.000  0.000
CNN       0.139  0.300  0.131
CNN-LSTM  0.000  0.000  0.000
Ensemble  0.069  0.000  0.069

✅ All models trained and evaluated. Now generating research figures..